In [4]:
import pandas as pd
import sqlite3

sessions = pd.read_csv(r"D:\clean_website_sessions.csv")
pageviews = pd.read_csv(r"D:\clean_website_pageviews.csv")
orders = pd.read_csv(r"D:\clean_orders.csv")
order_items = pd.read_csv(r"D:\clean_order_items.csv")
refunds = pd.read_csv(r"D:\clean_order_item_refunds.csv")
products = pd.read_csv(r"D:\clean_products.csv")

In [6]:
conn = sqlite3.connect(r"D:\maven_fuzzy_factory.db")

sessions.to_sql('website_sessions', conn, if_exists='replace', index=False)
pageviews.to_sql('website_pageviews', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
refunds.to_sql('order_item_refunds', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

4

In [66]:
query = """
SELECT s.utm_source, COUNT(DISTINCT s.website_session_id) AS sessions,
       COUNT(DISTINCT o.order_id) AS orders
FROM website_sessions s
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.utm_source
"""

result = pd.read_sql(query, conn)
result

,utm_source,sessions,orders
0,bsearch,62823,4519
1,direct,83328,6118
2,gsearch,316035,21333
3,socialbook,10685,343


In [39]:
query = """
SELECT device_type, COUNT(*) AS total_sessions
FROM website_sessions
GROUP BY device_type
"""
pd.read_sql(query, conn)

,device_type,total_sessions
0,desktop,327027
1,mobile,145844


In [40]:
query = """
SELECT is_repeat_session, COUNT(*) AS session_count
FROM website_sessions
GROUP BY is_repeat_session
"""
pd.read_sql(query, conn)

,is_repeat_session,session_count
0,0,394318
1,1,78553


In [41]:
query = """
SELECT strftime('%Y-%m', created_at) AS month, COUNT(*) AS sessions
FROM website_sessions
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,sessions
0,None,472871


In [42]:
query = """
SELECT utm_campaign, utm_content, COUNT(*) AS sessions
FROM website_sessions
GROUP BY utm_campaign, utm_content
ORDER BY sessions DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,utm_campaign,utm_content,sessions
0,nonbrand,g_ad_1,282706
1,none,none,83328
2,nonbrand,b_ad_1,54909
3,brand,g_ad_2,33329
4,brand,b_ad_2,7914
5,desktop_targeted,social_ad_2,5590
6,pilot,social_ad_1,5095


In [43]:
query = """
SELECT pageview_url, COUNT(*) AS views
FROM website_pageviews
GROUP BY pageview_url
ORDER BY views DESC
"""
pd.read_sql(query, conn)

,pageview_url,views
0,/products,230534
1,/the-original-mr-fuzzy,145884
2,/lander-2,131170
3,/home,119993
4,/cart,82603
5,/lander-3,69969
6,/shipping,56124
7,/lander-1,47574
8,/lander-5,43865
9,/billing-2,41696


In [44]:
query = """
SELECT website_session_id, COUNT(*) AS pages_viewed
FROM website_pageviews
GROUP BY website_session_id
ORDER BY pages_viewed DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,website_session_id,pages_viewed
0,421948,7
1,421937,7
2,421934,7
3,421928,7
4,421914,7
5,421913,7
6,421912,7
7,421898,7
8,421889,7
9,421883,7


In [45]:
query = """
SELECT pageview_url, COUNT(*) AS entry_count
FROM (
    SELECT website_session_id, pageview_url,
           ROW_NUMBER() OVER (PARTITION BY website_session_id ORDER BY created_at) AS rn
    FROM website_pageviews
) t
WHERE rn = 1
GROUP BY pageview_url
ORDER BY entry_count DESC
"""
pd.read_sql(query, conn)

,pageview_url,entry_count
0,/lander-2,131170
1,/home,119993
2,/lander-3,69969
3,/lander-1,47574
4,/lander-5,43865
5,/lander-4,9385


In [46]:
query = """
SELECT COUNT(*) AS total_orders, SUM(price_usd) AS total_revenue
FROM orders
"""
pd.read_sql(query, conn)

,total_orders,total_revenue
0,32313,1938509.75


In [47]:
query = """
SELECT strftime('%Y-%m', created_at) AS month,
       COUNT(*) AS orders, SUM(price_usd) AS revenue
FROM orders
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,orders,revenue
0,None,32313,1938509.75


In [48]:
query = """
SELECT AVG(price_usd) AS avg_order_value
FROM orders
"""
pd.read_sql(query, conn)

,avg_order_value
0,59.991636


In [49]:
query = """
SELECT items_purchased, COUNT(*) AS order_count
FROM orders
GROUP BY items_purchased
"""
pd.read_sql(query, conn)

,items_purchased,order_count
0,1,24601
1,2,7712


In [50]:
query = """
SELECT product_id,
       SUM(price_usd) AS total_revenue,
       SUM(cogs_usd) AS total_cost,
       SUM(price_usd - cogs_usd) AS total_profit
FROM order_items
GROUP BY product_id
ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,product_id,total_revenue,total_cost,total_profit
0,1,1211057.74,472164.74,738893.0
1,2,347702.04,130352.04,217350.0
2,3,229260.15,72232.65,157027.5
3,4,150489.82,47620.82,102869.0


In [51]:
query = """
SELECT product_id,
       ROUND(100.0 * SUM(price_usd - cogs_usd) / SUM(price_usd), 2) AS margin_pct
FROM order_items
GROUP BY product_id
"""
pd.read_sql(query, conn)

,product_id,margin_pct
0,1,61.01
1,2,62.51
2,3,68.49
3,4,68.36


In [52]:
query = """
SELECT COUNT(*) AS total_refunds, SUM(refund_amount_usd) AS total_refund_value
FROM order_item_refunds
"""
pd.read_sql(query, conn)

,total_refunds,total_refund_value
0,1731,85338.69


In [65]:
query = """
SELECT strftime('%Y-%m', created_at) AS month,
       COUNT(*) AS refunds, SUM(refund_amount_usd) AS refund_value
FROM order_item_refunds
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,refunds,refund_value
0,2012-04,5,249.95
1,2012-05,5,249.95
2,2012-06,5,249.95
3,2012-07,13,649.87
4,2012-08,18,899.82
5,2012-09,21,1049.79
6,2012-10,24,1199.76
7,2012-11,40,1999.60
8,2012-12,38,1899.62
9,2013-01,21,1059.79


In [54]:
query = """
SELECT product_id, product_name, created_at AS launched_on
FROM products
ORDER BY created_at
"""
pd.read_sql(query, conn)

,product_id,product_name,launched_on
0,1,The Original Mr. Fuzzy,2012-03-19 08:00:00
1,2,The Forever Love Bear,2013-01-06 13:00:00
2,3,The Birthday Sugar Panda,2013-12-12 09:00:00
3,4,The Hudson River Mini bear,2014-02-05 10:00:00


In [55]:
query = """
SELECT s.utm_source,
       COUNT(DISTINCT s.website_session_id) AS sessions,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT o.order_id) / COUNT(DISTINCT s.website_session_id), 2) AS conversion_rate_pct
FROM website_sessions s
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.utm_source
ORDER BY conversion_rate_pct DESC
"""
pd.read_sql(query, conn)

,utm_source,sessions,orders,conversion_rate_pct
0,direct,83328,6118,7.34
1,bsearch,62823,4519,7.19
2,gsearch,316035,21333,6.75
3,socialbook,10685,343,3.21


In [56]:
query = """
SELECT s.device_type,
       COUNT(DISTINCT s.website_session_id) AS sessions,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT o.order_id) / COUNT(DISTINCT s.website_session_id), 2) AS conversion_rate_pct
FROM website_sessions s
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.device_type
"""
pd.read_sql(query, conn)

,device_type,sessions,orders,conversion_rate_pct
0,desktop,327027,27805,8.50
1,mobile,145844,4508,3.09


In [57]:
query = """
SELECT p.product_name,
       SUM(oi.price_usd) AS total_revenue,
       SUM(oi.price_usd - oi.cogs_usd) AS total_profit
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,product_name,total_revenue,total_profit
0,The Original Mr. Fuzzy,1211057.74,738893.0
1,The Forever Love Bear,347702.04,217350.0
2,The Birthday Sugar Panda,229260.15,157027.5
3,The Hudson River Mini bear,150489.82,102869.0


In [58]:
query = """
WITH refund_counts AS (
    SELECT oi.product_id, COUNT(r.order_item_refund_id) AS refunds
    FROM order_items oi
    LEFT JOIN order_item_refunds r ON oi.order_item_id = r.order_item_id
    GROUP BY oi.product_id
)
SELECT p.product_name,
       COUNT(oi.order_item_id) AS items_sold,
       rc.refunds,
       ROUND(100.0 * rc.refunds / COUNT(oi.order_item_id), 2) AS refund_rate_pct
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN refund_counts rc ON oi.product_id = rc.product_id
GROUP BY p.product_name
"""
pd.read_sql(query, conn)

,product_name,items_sold,refunds,refund_rate_pct
0,The Birthday Sugar Panda,4985,301,6.04
1,The Forever Love Bear,5796,129,2.23
2,The Hudson River Mini bear,5018,64,1.28
3,The Original Mr. Fuzzy,24226,1237,5.11


In [59]:
query = """
SELECT order_id, created_at, price_usd,
       SUM(price_usd) OVER (ORDER BY created_at) AS running_revenue
FROM orders
ORDER BY created_at
"""
pd.read_sql(query, conn)

,order_id,created_at,price_usd,running_revenue
0,2587,01-01-2013 00:40,49.99,49.99
1,2588,01-01-2013 04:48,49.99,99.98
2,2589,01-01-2013 06:12,49.99,149.97
3,2590,01-01-2013 11:21,49.99,199.96
4,2591,01-01-2013 12:50,49.99,249.95
...,...,...,...,...
32308,26889,31-12-2014 19:23,49.99,1938263.80
32309,26890,31-12-2014 20:50,49.99,1938313.79
32310,26891,31-12-2014 21:44,49.99,1938363.78
32311,26892,31-12-2014 22:17,95.98,1938459.76


In [60]:
query = """
SELECT s.device_type,
       COUNT(DISTINCT pv.website_session_id) AS sessions_with_pageviews,
       COUNT(DISTINCT o.order_id) AS sessions_with_orders,
       ROUND(100.0 * COUNT(DISTINCT o.order_id) / COUNT(DISTINCT pv.website_session_id), 2) AS conversion_pct
FROM website_pageviews pv
JOIN website_sessions s ON pv.website_session_id = s.website_session_id
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.device_type
"""
pd.read_sql(query, conn)

,device_type,sessions_with_pageviews,sessions_with_orders,conversion_pct
0,desktop,291597,24017,8.24
1,mobile,130359,3956,3.03


In [61]:
query = """
SELECT s.utm_campaign,
       COUNT(DISTINCT s.website_session_id) AS sessions,
       SUM(o.price_usd) AS revenue,
       ROUND(SUM(o.price_usd) / COUNT(DISTINCT s.website_session_id), 2) AS revenue_per_session
FROM website_sessions s
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.utm_campaign
ORDER BY revenue DESC
"""
pd.read_sql(query, conn)

,utm_campaign,sessions,revenue,revenue_per_session
0,nonbrand,337615,1349977.69,4.00
1,none,83328,371433.03,4.46
2,brand,41243,194839.70,4.72
3,desktop_targeted,5590,18516.10,3.31
4,pilot,5095,3743.23,0.73


In [64]:
query = """
SELECT s.utm_source,
       COUNT(DISTINCT s.website_session_id) AS sessions,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(100.0 * COUNT(DISTINCT o.order_id) / COUNT(DISTINCT s.website_session_id), 2) AS conversion_rate_pct
FROM website_sessions s
LEFT JOIN orders o ON s.website_session_id = o.website_session_id
GROUP BY s.utm_source
ORDER BY conversion_rate_pct DESC
"""
channel_conversion = pd.read_sql(query, conn)
channel_conversion.to_csv(r"D:\powerbi_channel_conversion.csv", index=False)